# LT2 v6 - Liability Trading Algorithm

**Goal:** Unwind 50k shares (long or short) twice without moving the price.  
Block 1 arrives at tick ~0, Block 2 arrives at tick ~120. Must be flat by end.

**Key fixes from v5:**
1. `cap_order` now recognizes unwind orders as safe (was blocking ALL orders)
2. Emergency mode no longer traps at start — only fires for wrong-way exposure
3. Aggressive order protection logic corrected (more protection when spread is wide)
4. Clip sizes tuned for illiquid market — small passive clips, larger only on depth
5. TWAP schedule aligned with actual deadlines

**Strategy principles:**
1. PASSIVE-FIRST: post limit orders inside the spread to avoid impact
2. TWAP pacing: 50k / ~110 ticks = ~450 shares/tick target rate
3. Depth-aware: never take more than 40% of visible depth
4. Liquidity surge opportunism: hit book harder when depth is 2x+ average
5. Deadline escalation: get progressively more aggressive near block 2 and end
6. Spread-sensitive: wider spread = more passive, tighter spread = ok to cross

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import base64
import tkinter as tk
import statistics
from collections import deque

In [2]:
"""
RIT Gateway - API interface for the LT2 simulation.
"""


class ApiException(Exception):
    pass


class RITGateway:
    def __init__(self, username, password, host='http://141.211.79.101:10002', use_http=True):
        protocol = 'http' if use_http else 'https'
        self.base_url = f'{protocol}://{host.split("://")[-1]}/v1'
        self.session = requests.Session()

        auth_value = base64.b64encode(f'{username}:{password}'.encode()).decode()
        self.session.headers.update({
            'Authorization': f'Basic {auth_value}'
        })

    def _request(self, method, endpoint, params=None):
        url = f'{self.base_url}/{endpoint}'

        while True:
            if method == 'GET':
                resp = self.session.get(url, params=params)
            elif method == 'POST':
                resp = self.session.post(url, params=params)
            elif method == 'DELETE':
                resp = self.session.delete(url, params=params)
            else:
                raise ValueError(f'Unsupported HTTP method: {method}')

            if resp.status_code == 401:
                raise ApiException('Authentication failed.')

            if resp.status_code == 429:
                wait_time = float(resp.headers.get('Retry-After', 1))
                time.sleep(wait_time)
                continue

            if not resp.ok:
                raise ApiException(f'API request failed: {resp.status_code} {resp.text}')

            if resp.text.strip():
                return resp.json()
            return None

    def get_case(self):
        return self._request('GET', 'case')

    def get_securities(self):
        return self._request('GET', 'securities')

    def get_security(self, ticker):
        securities = self.get_securities()
        for sec in securities:
            if sec.get('ticker') == ticker:
                return sec
        return None

    def get_book(self, ticker, limit=200):
        return self._request('GET', 'securities/book', params={'ticker': ticker, 'limit': limit})

    def get_orders(self, status=None):
        params = {}
        if status is not None:
            params['status'] = status
        return self._request('GET', 'orders', params=params)

    def get_open_order_quantity(self, ticker):
        """Get total outstanding open order quantity by side for a ticker."""
        try:
            orders = self.get_orders(status='OPEN')
            if orders is None:
                return 0, 0
            buy_qty = 0
            sell_qty = 0
            for o in orders:
                if o.get('ticker') != ticker:
                    continue
                remaining = o.get('quantity', 0) - o.get('quantity_filled', 0)
                if o.get('action') == 'BUY':
                    buy_qty += remaining
                elif o.get('action') == 'SELL':
                    sell_qty += remaining
            return buy_qty, sell_qty
        except Exception:
            return 0, 0

    def submit_order(self, ticker, order_type, quantity, action, price=None):
        params = {
            'ticker': ticker,
            'type': order_type,
            'quantity': int(quantity),
            'action': action
        }
        if price is not None:
            params['price'] = round(float(price), 2)

        return self._request('POST', 'orders', params=params)

    def cancel_all_orders(self):
        return self._request('POST', 'commands/cancel', params={'all': 1})

    def cancel_order(self, order_id):
        return self._request('DELETE', f'orders/{order_id}')

In [3]:
class MarketMonitor:
    """Tracks recent market microstructure for decision-making."""

    def __init__(
        self,
        max_history=60,
        wide_spread_threshold=0.06,
        thin_depth_threshold=1500,
        imbalance_threshold=0.55,
        adverse_move_threshold=0.03,
        shock_move_threshold=0.08,
        depth_surge_multiplier=2.0,
    ):
        self.max_history = max_history
        self.wide_spread_threshold = wide_spread_threshold
        self.thin_depth_threshold = thin_depth_threshold
        self.imbalance_threshold = imbalance_threshold
        self.adverse_move_threshold = adverse_move_threshold
        self.shock_move_threshold = shock_move_threshold
        self.depth_surge_multiplier = depth_surge_multiplier

        self.recent_spreads = deque(maxlen=max_history)
        self.recent_bid_depth = deque(maxlen=max_history)
        self.recent_ask_depth = deque(maxlen=max_history)
        self.recent_mid = deque(maxlen=max_history)
        self.recent_bid_depth_3lvl = deque(maxlen=max_history)
        self.recent_ask_depth_3lvl = deque(maxlen=max_history)

    def record_book(self, best_bid, best_ask, bid_sizes=None, ask_sizes=None):
        spread = round(best_ask - best_bid, 4)
        self.recent_spreads.append(spread)

        top_bid = bid_sizes[0] if bid_sizes and len(bid_sizes) > 0 else 0
        top_ask = ask_sizes[0] if ask_sizes and len(ask_sizes) > 0 else 0

        self.recent_bid_depth.append(top_bid)
        self.recent_ask_depth.append(top_ask)
        self.recent_mid.append((best_bid + best_ask) / 2)

        bid_3 = sum(bid_sizes[:3]) if bid_sizes else 0
        ask_3 = sum(ask_sizes[:3]) if ask_sizes else 0
        self.recent_bid_depth_3lvl.append(bid_3)
        self.recent_ask_depth_3lvl.append(ask_3)

    def current_spread(self):
        return self.recent_spreads[-1] if self.recent_spreads else 0.05

    def avg_spread(self):
        if len(self.recent_spreads) < 3:
            return 0.05
        return statistics.mean(self.recent_spreads)

    def detect_thin_liquidity(self, side=None):
        if side == 'SELL':
            if not self.recent_bid_depth:
                return True
            return self.recent_bid_depth[-1] < self.thin_depth_threshold
        if side == 'BUY':
            if not self.recent_ask_depth:
                return True
            return self.recent_ask_depth[-1] < self.thin_depth_threshold
        if not self.recent_bid_depth or not self.recent_ask_depth:
            return True
        return (
            self.recent_bid_depth[-1] < self.thin_depth_threshold
            or self.recent_ask_depth[-1] < self.thin_depth_threshold
        )

    def book_imbalance(self):
        """Positive = more bids, negative = more asks."""
        if not self.recent_bid_depth_3lvl or not self.recent_ask_depth_3lvl:
            return 0.0
        bid_d = self.recent_bid_depth_3lvl[-1]
        ask_d = self.recent_ask_depth_3lvl[-1]
        total = bid_d + ask_d
        if total <= 0:
            return 0.0
        return (bid_d - ask_d) / total

    def favorable_for_selling(self):
        """Strong bid side = good time to sell into bids."""
        return self.book_imbalance() >= self.imbalance_threshold

    def favorable_for_buying(self):
        """Strong ask side = good time to buy (plenty of offers to lift)."""
        return self.book_imbalance() <= -self.imbalance_threshold

    def recent_move(self, lookback=1):
        if len(self.recent_mid) <= lookback:
            return 0.0
        return self.recent_mid[-1] - self.recent_mid[-1 - lookback]

    def adverse_move(self, inventory, lookback=1):
        """Price moved against our position."""
        move = self.recent_move(lookback=lookback)
        if inventory > 0:
            return move <= -self.adverse_move_threshold
        if inventory < 0:
            return move >= self.adverse_move_threshold
        return False

    def price_is_favorable(self, inventory):
        """Price moving in our favor = can be more patient."""
        move_1 = self.recent_move(lookback=1)
        move_3 = self.recent_move(lookback=3)
        threshold = 0.02
        if inventory > 0 and (move_1 > threshold or move_3 > threshold * 2):
            return True
        if inventory < 0 and (move_1 < -threshold or move_3 < -threshold * 2):
            return True
        return False

    def detect_liquidity_surge(self, inventory):
        """
        Detect when depth on our unwind side is significantly above average.
        This is the ideal moment to be aggressive.
        Returns: (is_surge, current_depth, avg_depth)
        """
        if inventory > 0:
            depths = self.recent_bid_depth_3lvl
        elif inventory < 0:
            depths = self.recent_ask_depth_3lvl
        else:
            return False, 0, 0

        if len(depths) < 5:
            return False, 0, 0

        current = depths[-1]
        avg = statistics.mean(list(depths)[:-1])

        if avg <= 0:
            return False, current, 0

        is_surge = current >= avg * self.depth_surge_multiplier
        return is_surge, current, avg

    def reset(self):
        self.recent_spreads.clear()
        self.recent_bid_depth.clear()
        self.recent_ask_depth.clear()
        self.recent_mid.clear()
        self.recent_bid_depth_3lvl.clear()
        self.recent_ask_depth_3lvl.clear()

In [4]:
class TWAPScheduler:
    """Tracks how far behind/ahead of a linear unwind schedule we are."""

    def __init__(
        self,
        start_tick=0,
        end_tick=110,
        catchup_sensitivity=1.2,
        min_progress_buffer=1500
    ):
        self.start_tick = start_tick
        self.end_tick = end_tick
        self.catchup_sensitivity = catchup_sensitivity
        self.min_progress_buffer = min_progress_buffer
        self.initial_inventory = None
        self.side = None

    def reset(self):
        self.initial_inventory = None
        self.side = None

    def initialize(self, inventory):
        if inventory == 0:
            return
        if self.initial_inventory is None:
            self.initial_inventory = abs(inventory)
            self.side = 'LONG' if inventory > 0 else 'SHORT'

    def reinitialize(self, inventory, new_start, new_end):
        self.start_tick = new_start
        self.end_tick = new_end
        self.initial_inventory = abs(inventory)
        self.side = 'LONG' if inventory > 0 else 'SHORT'

    def progress_ratio(self, elapsed):
        if elapsed <= self.start_tick:
            return 0.0
        if elapsed >= self.end_tick:
            return 1.0
        return (elapsed - self.start_tick) / max(1, self.end_tick - self.start_tick)

    def target_remaining(self, elapsed):
        if self.initial_inventory is None:
            return 0
        progress = self.progress_ratio(elapsed)
        remaining = self.initial_inventory * (1.0 - progress)
        return max(0, int(round(remaining / 100.0)) * 100)

    def shares_per_tick(self):
        if self.initial_inventory is None:
            return 500
        total_ticks = max(1, self.end_tick - self.start_tick)
        return max(200, int(self.initial_inventory / total_ticks))

    def schedule_gap(self, inventory, elapsed):
        """Positive = behind schedule, negative = ahead."""
        if self.initial_inventory is None:
            self.initialize(inventory)
        actual_remaining = abs(inventory)
        target = self.target_remaining(elapsed)
        return actual_remaining - target

    def behind_schedule(self, inventory, elapsed):
        return self.schedule_gap(inventory, elapsed) > self.min_progress_buffer

    def ahead_of_schedule(self, inventory, elapsed):
        return self.schedule_gap(inventory, elapsed) < -500

    def urgency_boost(self, inventory, elapsed):
        if self.initial_inventory is None:
            self.initialize(inventory)
        if self.initial_inventory is None or self.initial_inventory == 0:
            return 0.0
        gap = max(0, self.schedule_gap(inventory, elapsed))
        boost = (gap / self.initial_inventory) * self.catchup_sensitivity
        return min(1.0, max(0.0, boost))

    def target_clip(self, inventory, elapsed, base_clip=400, max_clip=2500):
        if self.initial_inventory is None:
            self.initialize(inventory)
        spt = self.shares_per_tick()
        clip = max(base_clip, spt)

        gap = max(0, self.schedule_gap(inventory, elapsed))
        if gap > 0:
            clip = clip + int(0.3 * gap)

        clip = min(max_clip, clip)
        clip = min(abs(inventory), clip)
        clip = int(clip / 100) * 100
        return max(0, clip)

In [5]:
class Trader:
    """
    Main decision engine for unwinding block positions in illiquid markets.

    The core challenge: we get 50k shares dumped on us and must unwind
    without moving the price. ANON flow is tiny, so our orders ARE the
    market. Every aggressive order moves the price against us.

    Approach:
    - Default to passive limit orders inside the spread
    - Only cross the spread when: deadline pressure, liquidity surge,
      or book imbalance strongly in our favor
    - Keep clip sizes small relative to visible depth
    - Escalate aggression only as deadlines approach
    """

    def __init__(
        self,
        monitor,
        twap=None,
        position_limit=50000,
        second_block_time=120,
        hard_flatten_time=225,
        final_flatten_time=280,
        pre_second_safe_inventory=2000,
        pre_second_window_1=30,   # ticks before block: start worrying
        pre_second_window_2=15,   # ticks before block: get aggressive
        pre_second_window_3=5,    # ticks before block: emergency
        patience_ticks=3,
        wide_spread_threshold=0.06,
    ):
        self.monitor = monitor
        self.twap = twap
        self.position_limit = position_limit
        self.second_block_time = second_block_time
        self.hard_flatten_time = hard_flatten_time
        self.final_flatten_time = final_flatten_time
        self.pre_second_safe_inventory = pre_second_safe_inventory
        self.pre_second_window_1 = pre_second_window_1
        self.pre_second_window_2 = pre_second_window_2
        self.pre_second_window_3 = pre_second_window_3
        self.patience_ticks = patience_ticks
        self.wide_spread_threshold = wide_spread_threshold

        # State tracking
        self.last_abs_inventory = None
        self.last_block_tick = None
        self.second_block_detected = False
        self.block_entry_price = None
        self.initial_mid = None
        self.unwind_side = None  # 'BUY' or 'SELL' — direction we need to trade

    # =========================================================
    # Phase detection
    # =========================================================
    def detect_phase(self, elapsed):
        if elapsed is None:
            return 'PRE-SECOND-BLOCK'
        if elapsed < self.second_block_time:
            return 'PRE-SECOND-BLOCK'
        if elapsed < self.hard_flatten_time:
            return 'POST-SECOND-BLOCK'
        if elapsed < self.final_flatten_time:
            return 'HARD-FLATTEN'
        return 'FINAL-FLATTEN'

    # =========================================================
    # Block detection
    # =========================================================
    def update_block_state(self, inventory, elapsed, mid_price):
        abs_inv = abs(inventory)
        if self.initial_mid is None and mid_price:
            self.initial_mid = mid_price

        # Track which direction we need to trade to unwind
        if inventory != 0:
            self.unwind_side = 'SELL' if inventory > 0 else 'BUY'

        if self.last_abs_inventory is not None and elapsed is not None:
            jump = abs_inv - self.last_abs_inventory

            # First block detection (happens at start)
            if self.block_entry_price is None and abs_inv >= 40000 and elapsed <= 3:
                self.block_entry_price = mid_price
                if self.twap:
                    self.twap.initialize(inventory)
                print(f"[BLOCK] First block tick={elapsed} inv={inventory} mid={mid_price:.2f}")

            # Second block detection
            if elapsed >= self.second_block_time and jump >= 25000 and not self.second_block_detected:
                self.last_block_tick = elapsed
                self.second_block_detected = True
                self.block_entry_price = mid_price
                if self.twap:
                    self.twap.reinitialize(inventory, new_start=elapsed, new_end=self.hard_flatten_time)
                print(f"[BLOCK] Second block tick={elapsed} inv={inventory} mid={mid_price:.2f}")

        self.last_abs_inventory = abs_inv

    # =========================================================
    # Urgency computation
    # =========================================================
    def compute_urgency(self, inventory, elapsed):
        abs_inv = abs(inventory)
        inv_ratio = min(1.0, abs_inv / max(1, self.position_limit))

        if elapsed is None:
            base = 0.15 * inv_ratio
        elif elapsed < self.second_block_time:
            time_left = self.second_block_time - elapsed
            progress = elapsed / max(1, self.second_block_time)
            base = 0.10 * inv_ratio + 0.05 * progress

            # Escalate as second block approaches
            if abs_inv > self.pre_second_safe_inventory:
                excess = min(1.0, (abs_inv - self.pre_second_safe_inventory) / max(1, self.position_limit))
                base += 0.10 * excess

            if time_left <= self.pre_second_window_1:
                base += 0.08
            if time_left <= self.pre_second_window_2:
                base += 0.12
            if time_left <= self.pre_second_window_3:
                base += 0.25

        elif elapsed < self.hard_flatten_time:
            progress = (elapsed - self.second_block_time) / max(1, self.hard_flatten_time - self.second_block_time)
            base = 0.20 * inv_ratio + 0.10 * progress + 0.05

        elif elapsed < self.final_flatten_time:
            progress = (elapsed - self.hard_flatten_time) / max(1, self.final_flatten_time - self.hard_flatten_time)
            base = 0.40 * inv_ratio + 0.20 * progress + 0.10
        else:
            base = 1.0

        # Micro-adjustments
        if self.monitor.adverse_move(inventory, lookback=1):
            base += 0.06
        if self.monitor.price_is_favorable(inventory):
            base -= 0.10
        if self.twap and elapsed is not None:
            base += 0.15 * self.twap.urgency_boost(inventory, elapsed)

        return min(1.0, max(0.0, base))

    # =========================================================
    # Clip sizing
    # =========================================================
    def choose_clip_size(self, inventory, elapsed, bid_sizes=None, ask_sizes=None, surge=False):
        """
        Key insight: in an illiquid market, small clips are everything.
        We want to drip-feed orders so we don't move the price.

        50k / 110 ticks = ~450 shares/tick.
        With ~3 polls per tick at 0.25s delay = ~150 shares per poll.
        But we use 200-500 as base range since not every poll leads to a fill.
        """
        abs_inv = abs(inventory)
        if abs_inv == 0:
            return 0

        # Base from TWAP
        if self.twap and elapsed is not None:
            twap_clip = self.twap.target_clip(inventory, elapsed, base_clip=300, max_clip=2000)
        else:
            twap_clip = 400

        clip = twap_clip

        # Behind schedule: allow larger
        if self.twap and elapsed is not None:
            gap = self.twap.schedule_gap(inventory, elapsed)
            if gap > 5000:
                clip = min(2000, clip + 600)
            if gap > 15000:
                clip = min(3500, clip + 1200)

        # LIQUIDITY SURGE: 2x normal clip
        if surge:
            clip = min(abs_inv, int(clip * 2.0))
            clip = min(clip, 4000)

        # DEPTH-AWARE: never take more than 40% of visible depth on our unwind side
        if inventory > 0:
            sizes = bid_sizes or []
        else:
            sizes = ask_sizes or []
        depth = sum(sizes[:3])
        if depth > 0:
            clip = min(clip, max(200, int(depth * 0.40)))

        # Near deadlines, override depth constraint
        if elapsed is not None:
            if elapsed < self.second_block_time:
                time_left = self.second_block_time - elapsed
                if time_left <= self.pre_second_window_3 and abs_inv > 500:
                    clip = min(abs_inv, max(clip, 5000))
                elif time_left <= self.pre_second_window_2 and abs_inv > 2000:
                    clip = min(abs_inv, max(clip, 3000))
            elif elapsed >= self.final_flatten_time:
                clip = min(abs_inv, max(clip, 5000))

        clip = min(abs_inv, clip)
        clip = max(200, int(clip / 100) * 100)
        return clip

    # =========================================================
    # Order generation
    # =========================================================
    def make_passive_order(self, best_bid, best_ask, inventory, urgency, elapsed,
                           bid_sizes=None, ask_sizes=None, post_at_inside=False):
        """
        Generate a passive limit order.

        The key to not moving the price: post ON our unwind side of the book.
        If we're long and need to sell, we post on the ask side.
        If we're short and need to buy, we post on the bid side.

        This way we get filled by incoming ANON flow without crossing the spread.
        """
        if best_bid is None or best_ask is None or best_ask <= best_bid:
            return None

        abs_inv = abs(inventory)
        spread = best_ask - best_bid
        mid = (best_bid + best_ask) / 2

        qty = self.choose_clip_size(inventory, elapsed, bid_sizes, ask_sizes)
        qty = min(qty, abs_inv)
        if qty < 100:
            return None

        if inventory > 0:
            # SELLING to unwind long: post on the ask side
            if post_at_inside:
                # Post AT best ask — be first in line for incoming buys
                px = best_ask
            elif urgency >= 0.60:
                # Urgent: post at mid or just above bid to get filled faster
                px = round(mid, 2)
            elif spread > self.wide_spread_threshold:
                # Wide spread: post inside spread to capture half-spread edge
                px = round(mid + 0.01, 2)
                px = max(px, best_bid + 0.02)
            else:
                # Tight spread: post just below best ask
                px = round(best_ask - 0.01, 2)

            # Safety: never post below best bid (would be giving money away)
            px = max(px, best_bid + 0.01)
            return {'action': 'SELL', 'quantity': qty, 'price': round(px, 2)}

        if inventory < 0:
            # BUYING to unwind short: post on the bid side
            if post_at_inside:
                px = best_bid
            elif urgency >= 0.60:
                px = round(mid, 2)
            elif spread > self.wide_spread_threshold:
                px = round(mid - 0.01, 2)
                px = min(px, best_ask - 0.02)
            else:
                px = round(best_bid + 0.01, 2)

            # Safety: never post above best ask
            px = min(px, best_ask - 0.01)
            return {'action': 'BUY', 'quantity': qty, 'price': round(px, 2)}

        return None

    def make_aggressive_order(self, best_bid, best_ask, inventory, qty, spread=None):
        """
        Marketable limit order — crosses the spread to get filled immediately.

        Protection logic: wider spread = MORE protection (not less!)
        because wide spreads mean thin books and higher slippage risk.
        """
        if qty < 100:
            return None

        # Protection: how far past the opposite side we're willing to fill
        if spread is not None and spread > 0.15:
            # Very wide spread: tight protection, we're already paying a lot
            protection = 0.03
        elif spread is not None and spread > 0.08:
            protection = 0.04
        else:
            # Tight spread: can afford a bit more protection
            protection = 0.05

        if inventory > 0:
            # Selling: marketable limit below best bid
            price = round(best_bid - protection, 2)
            return {'action': 'SELL', 'quantity': qty, 'price': price}
        elif inventory < 0:
            # Buying: marketable limit above best ask
            price = round(best_ask + protection, 2)
            return {'action': 'BUY', 'quantity': qty, 'price': price}
        return None

    # =========================================================
    # Main decision engine
    # =========================================================
    def decide(self, best_bid, best_ask, inventory, elapsed=None,
               bid_sizes=None, ask_sizes=None):

        mid_price = (best_bid + best_ask) / 2
        spread = round(best_ask - best_bid, 4)
        self.update_block_state(inventory, elapsed, mid_price)
        self.monitor.record_book(best_bid, best_ask, bid_sizes=bid_sizes, ask_sizes=ask_sizes)

        phase = self.detect_phase(elapsed)
        urgency = self.compute_urgency(inventory, elapsed)
        imbalance = self.monitor.book_imbalance()
        is_surge, surge_depth, avg_depth = self.monitor.detect_liquidity_surge(inventory)
        wide_spread = spread >= self.wide_spread_threshold

        # === FLAT: nothing to do ===
        if inventory == 0:
            return {
                'phase': phase, 'mode': 'Flat', 'urgency': 0,
                'imbalance': imbalance, 'order': None, 'is_aggressive': False,
                'alert_text': 'FLAT', 'alert_bg': 'dark green'
            }

        abs_inv = abs(inventory)

        # === 1) WRONG-WAY EMERGENCY ===
        # Only trigger if we're OVER the gross limit (shouldn't happen, but safety net)
        # This is NOT for normal unwind — that's handled below.
        if abs_inv > self.position_limit:
            qty = min(5000, abs_inv - self.position_limit + 1000)
            qty = max(200, int(qty / 100) * 100)
            order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
            side = 'Sell' if inventory > 0 else 'Buy'
            return {
                'phase': phase, 'mode': f'Emergency {side}', 'urgency': 1.0,
                'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                'alert_text': f'OVER LIMIT - {side.upper()}', 'alert_bg': 'red'
            }

        # === 2) FINAL FLATTEN: time is almost up ===
        if elapsed is not None and elapsed >= self.final_flatten_time and abs_inv > 0:
            qty = min(abs_inv, 5000)
            order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
            side = 'Sell' if inventory > 0 else 'Buy'
            return {
                'phase': phase, 'mode': f'Final Flatten {side}', 'urgency': 1.0,
                'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                'alert_text': 'FINAL FLATTEN', 'alert_bg': 'red'
            }

        # === 3) Pre-second-block deadline pressure ===
        if elapsed is not None and elapsed < self.second_block_time:
            time_left = self.second_block_time - elapsed

            if time_left <= self.pre_second_window_3 and abs_inv > 500:
                qty = min(abs_inv, 5000)
                order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
                return {
                    'phase': phase, 'mode': 'Pre-Block Emergency', 'urgency': min(1.0, urgency + 0.3),
                    'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                    'alert_text': 'BLOCK IMMINENT', 'alert_bg': 'red'
                }

            if time_left <= self.pre_second_window_2 and abs_inv > 2000:
                qty = min(abs_inv, 3000)
                order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
                return {
                    'phase': phase, 'mode': 'Pre-Block Forced', 'urgency': min(1.0, urgency + 0.2),
                    'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                    'alert_text': 'COUNTDOWN FLATTEN', 'alert_bg': 'orange'
                }

        # === 4) HARD FLATTEN deadline pressure ===
        if elapsed is not None and elapsed >= self.hard_flatten_time and elapsed < self.final_flatten_time:
            if abs_inv > 5000:
                qty = self.choose_clip_size(inventory, elapsed, bid_sizes, ask_sizes)
                qty = min(abs_inv, max(qty, 2000))
                order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
                side = 'Sell' if inventory > 0 else 'Buy'
                return {
                    'phase': phase, 'mode': f'Hard Flatten {side}', 'urgency': urgency,
                    'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                    'alert_text': 'HARD FLATTEN', 'alert_bg': 'orange'
                }

        # === 5) Patience window: just received a block, let it settle ===
        in_patience = False
        if elapsed is not None and elapsed <= self.patience_ticks:
            in_patience = True
        if self.last_block_tick is not None and elapsed is not None:
            if elapsed - self.last_block_tick <= self.patience_ticks:
                in_patience = True

        if in_patience and abs_inv <= self.position_limit:
            order = self.make_passive_order(
                best_bid, best_ask, inventory,
                urgency=min(urgency, 0.20), elapsed=elapsed,
                bid_sizes=bid_sizes, ask_sizes=ask_sizes,
                post_at_inside=True
            )
            return {
                'phase': phase, 'mode': 'Patience', 'urgency': urgency,
                'imbalance': imbalance, 'order': order, 'is_aggressive': False,
                'alert_text': 'SETTLING', 'alert_bg': 'dark green'
            }

        # === 6) LIQUIDITY SURGE: depth is 2x+ average — hit it ===
        if is_surge and abs_inv > 1000 and not wide_spread:
            qty = self.choose_clip_size(inventory, elapsed, bid_sizes, ask_sizes, surge=True)
            order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
            return {
                'phase': phase, 'mode': 'Liquidity Surge', 'urgency': urgency,
                'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                'alert_text': f'SURGE depth={surge_depth} avg={avg_depth:.0f}', 'alert_bg': 'dark blue'
            }

        # === 7) Wide spread: stay passive, capture edge ===
        if wide_spread and urgency < 0.60:
            order = self.make_passive_order(
                best_bid, best_ask, inventory,
                urgency=urgency, elapsed=elapsed,
                bid_sizes=bid_sizes, ask_sizes=ask_sizes
            )
            return {
                'phase': phase, 'mode': 'Wide Spread Passive', 'urgency': urgency,
                'imbalance': imbalance, 'order': order, 'is_aggressive': False,
                'alert_text': 'WIDE SPREAD', 'alert_bg': 'purple'
            }

        # === 8) Favorable price: be patient ===
        if self.monitor.price_is_favorable(inventory) and abs_inv < 40000:
            order = self.make_passive_order(
                best_bid, best_ask, inventory,
                urgency=max(0.10, urgency - 0.10), elapsed=elapsed,
                bid_sizes=bid_sizes, ask_sizes=ask_sizes,
                post_at_inside=True
            )
            return {
                'phase': phase, 'mode': 'Favorable Patient', 'urgency': urgency,
                'imbalance': imbalance, 'order': order, 'is_aggressive': False,
                'alert_text': 'FAVORABLE', 'alert_bg': 'dark green'
            }

        # === 9) Book imbalance in our favor: small opportunistic aggression ===
        if inventory > 0 and self.monitor.favorable_for_selling() and not self.monitor.detect_thin_liquidity('SELL'):
            qty = min(1500, self.choose_clip_size(inventory, elapsed, bid_sizes, ask_sizes))
            order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
            return {
                'phase': phase, 'mode': 'Opportunistic Sell', 'urgency': urgency,
                'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                'alert_text': 'STRONG BID', 'alert_bg': 'dark blue'
            }
        if inventory < 0 and self.monitor.favorable_for_buying() and not self.monitor.detect_thin_liquidity('BUY'):
            qty = min(1500, self.choose_clip_size(inventory, elapsed, bid_sizes, ask_sizes))
            order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
            return {
                'phase': phase, 'mode': 'Opportunistic Buy', 'urgency': urgency,
                'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                'alert_text': 'STRONG ASK', 'alert_bg': 'dark blue'
            }

        # === 10) Behind TWAP schedule: controlled aggression ===
        if self.twap and elapsed is not None and self.twap.behind_schedule(inventory, elapsed):
            gap = self.twap.schedule_gap(inventory, elapsed)
            if gap > 3000 and not self.monitor.detect_thin_liquidity():
                qty = self.choose_clip_size(inventory, elapsed, bid_sizes, ask_sizes)
                order = self.make_aggressive_order(best_bid, best_ask, inventory, qty, spread)
                side = 'Sell' if inventory > 0 else 'Buy'
                return {
                    'phase': phase, 'mode': f'TWAP Catchup {side}', 'urgency': urgency,
                    'imbalance': imbalance, 'order': order, 'is_aggressive': True,
                    'alert_text': f'BEHIND SCHED gap={gap}', 'alert_bg': 'dark orange'
                }

        # === 11) Default: passive unwind at TWAP pace ===
        ahead = self.twap and elapsed is not None and self.twap.ahead_of_schedule(inventory, elapsed)
        order = self.make_passive_order(
            best_bid, best_ask, inventory,
            urgency=urgency, elapsed=elapsed,
            bid_sizes=bid_sizes, ask_sizes=ask_sizes,
            post_at_inside=ahead
        )

        alert_text = 'PASSIVE UNWIND'
        alert_bg = 'dark green'
        if self.monitor.detect_thin_liquidity():
            alert_text = 'THIN BOOK'
            alert_bg = 'brown'

        return {
            'phase': phase, 'mode': 'Passive Unwind', 'urgency': urgency,
            'imbalance': imbalance, 'order': order, 'is_aggressive': False,
            'alert_text': alert_text, 'alert_bg': alert_bg
        }

In [6]:
class TraderUI:
    _existing_root = None

    def __init__(self):
        if TraderUI._existing_root is not None:
            try:
                TraderUI._existing_root.destroy()
            except Exception:
                pass
            TraderUI._existing_root = None

        self.root = tk.Tk()
        TraderUI._existing_root = self.root
        self.root.title('LT2 v6 Dashboard')
        self.root.geometry('650x520')
        self.root.configure(bg='black')

        self.gateway = None
        self.stop_requested = False

        self.title = tk.Label(self.root, text='LT2 v6 TRADER', font=('Helvetica', 18, 'bold'), fg='white', bg='black')
        self.title.pack(pady=6)

        self.position_label = self._label('Position: 0')
        self.pnl_label = self._label('PnL: 0.00')
        self.phase_label = self._label('Phase: -')
        self.mode_label = self._label('Mode: -')
        self.market_label = self._label('Bid / Ask: - / -')
        self.depth_label = self._label('Bid/Ask Depth(3lvl): - / -')
        self.urgency_label = self._label('Urgency: 0.00')
        self.imbalance_label = self._label('Imbalance: 0.00')
        self.twap_label = self._label('TWAP Gap: 0')
        self.order_label = self._label('Order: None')

        self.alert_label = tk.Label(self.root, text='NORMAL', font=('Helvetica', 14, 'bold'),
                                     fg='white', bg='dark green', width=30, height=2)
        self.alert_label.pack(pady=8)

        btn_frame = tk.Frame(self.root, bg='black')
        btn_frame.pack(pady=8)

        tk.Button(btn_frame, text='KILL ALL', command=self.kill_all_orders,
                  font=('Helvetica', 12, 'bold'), fg='white', bg='red', width=14).pack(side='left', padx=6)
        tk.Button(btn_frame, text='STOP LOOP', command=self.request_stop,
                  font=('Helvetica', 12, 'bold'), fg='white', bg='gray25', width=14).pack(side='left', padx=6)

        self.root.protocol('WM_DELETE_WINDOW', self.on_close)

    def _label(self, text):
        lbl = tk.Label(self.root, text=text, font=('Helvetica', 12), fg='white', bg='black')
        lbl.pack(pady=3)
        return lbl

    def set_gateway(self, gateway):
        self.gateway = gateway

    def kill_all_orders(self):
        if self.gateway:
            try:
                self.gateway.cancel_all_orders()
                self.alert_label.config(text='KILLED ALL ORDERS', bg='red')
            except Exception as e:
                self.alert_label.config(text=f'KILL FAILED: {e}', bg='maroon')
        self.refresh()

    def request_stop(self):
        self.stop_requested = True
        self.alert_label.config(text='STOP REQUESTED', bg='gray25')
        self.refresh()

    def on_close(self):
        self.stop_requested = True
        try:
            self.root.destroy()
        except Exception:
            pass
        TraderUI._existing_root = None

    def refresh(self):
        try:
            self.root.update_idletasks()
            self.root.update()
        except tk.TclError:
            pass

    def update(self, inventory, pnl, phase, mode, best_bid, best_ask,
               bid_depth_3, ask_depth_3, urgency=0.0, imbalance=0.0,
               twap_gap=0, working_order=None,
               alert_text='NORMAL', alert_bg='dark green'):
        self.position_label.config(text=f'Position: {inventory}')
        self.pnl_label.config(text=f'PnL: {pnl:.2f}')
        self.phase_label.config(text=f'Phase: {phase}')
        self.mode_label.config(text=f'Mode: {mode}')
        self.market_label.config(text=f'Bid / Ask: {best_bid:.2f} / {best_ask:.2f}')
        self.depth_label.config(text=f'Bid/Ask Depth(3lvl): {bid_depth_3} / {ask_depth_3}')
        self.urgency_label.config(text=f'Urgency: {urgency:.2f}')
        self.imbalance_label.config(text=f'Imbalance: {imbalance:.2f}')
        self.twap_label.config(text=f'TWAP Gap: {twap_gap}')

        if working_order is None:
            self.order_label.config(text='Order: None')
        else:
            self.order_label.config(
                text=f"Order: {working_order['action']} {working_order['quantity']} @ {working_order['price']:.2f}"
            )

        self.alert_label.config(text=alert_text, bg=alert_bg)
        self.refresh()

In [7]:
# =========================================================
# Logging and analysis utilities
# =========================================================


def log_loop(loop_log, tick, best_bid, best_ask, inventory, pnl, decision):
    order = decision.get('order')
    row = {
        'tick': tick,
        'best_bid': best_bid,
        'best_ask': best_ask,
        'mid': (best_bid + best_ask) / 2,
        'spread': round(best_ask - best_bid, 4),
        'inventory': inventory,
        'pnl': pnl,
        'mode': decision.get('mode'),
        'urgency': decision.get('urgency', 0),
        'imbalance': decision.get('imbalance', 0),
        'phase': decision.get('phase'),
        'is_aggressive': decision.get('is_aggressive', False),
        'order_action': order.get('action') if order else None,
        'order_qty': order.get('quantity') if order else None,
        'order_price': order.get('price') if order else None,
    }
    loop_log.append(row)


def log_order(order_log, tick, ticker, event_type, mode, order_type, action, quantity, price, inventory_before):
    order_log.append({
        'tick': tick, 'ticker': ticker, 'event_type': event_type, 'mode': mode,
        'order_type': order_type, 'action': action, 'quantity': quantity,
        'price': price, 'inventory_before': inventory_before
    })


def infer_and_log_fills(fill_log, tick, prev_inventory, inventory, best_bid, best_ask, prev_mid, mode):
    delta = inventory - prev_inventory
    if delta == 0:
        return
    side = 'BUY' if delta > 0 else 'SELL'
    qty = abs(delta)
    aggressive = mode and any(k in mode for k in ['Emergency', 'Forced', 'Surge', 'Catchup', 'Opportunistic', 'Hard', 'Final'])
    if side == 'BUY':
        est_price = best_ask if aggressive else best_bid
        edge = prev_mid - est_price
    else:
        est_price = best_bid if aggressive else best_ask
        edge = est_price - prev_mid
    fill_log.append({
        'tick': tick, 'side': side, 'qty': qty,
        'inv_before': prev_inventory, 'inv_after': inventory,
        'est_price': est_price, 'mid_before': prev_mid,
        'edge_per_share': edge, 'est_edge_dollars': edge * qty, 'mode': mode
    })


def analyze_run(loop_df, order_df, fill_df, second_block_time=120, save_prefix=None):
    print('=' * 60)
    print('                    RUN ANALYSIS')
    print('=' * 60)

    if loop_df.empty:
        print('No data to analyze.')
        return

    final = loop_df.iloc[-1]
    print(f"Final tick: {final['tick']}")
    print(f"Final PnL: {final['pnl']:.2f}")
    print(f"Final inventory: {final['inventory']}")
    print(f"Max abs inventory: {loop_df['inventory'].abs().max()}")
    print(f"Avg abs inventory: {loop_df['inventory'].abs().mean():.0f}")
    print(f"Time with |inv| > 3000: {(loop_df['inventory'].abs() > 3000).sum()} rows")
    print()

    pre = loop_df[loop_df['tick'] < second_block_time]
    post = loop_df[loop_df['tick'] >= second_block_time]
    if not pre.empty:
        print(f"Pre-block final PnL: {pre.iloc[-1]['pnl']:.2f}")
        print(f"Pre-block max abs inv: {pre['inventory'].abs().max()}")
    if not post.empty:
        print(f"Post-block final PnL: {post.iloc[-1]['pnl']:.2f}")
        print(f"Post-block max abs inv: {post['inventory'].abs().max()}")

    at_block = loop_df[loop_df['tick'] >= second_block_time]
    if not at_block.empty:
        print(f"Inv at second block: {at_block.iloc[0]['inventory']}")
        print(f"PnL at second block: {at_block.iloc[0]['pnl']:.2f}")
    print()

    if 'spread' in loop_df.columns:
        print(f"Avg spread: {loop_df['spread'].mean():.4f}")
        print(f"Max spread: {loop_df['spread'].max():.4f}")
        print(f"Pct time spread > 0.10: {(loop_df['spread'] > 0.10).mean()*100:.1f}%")
    print()

    if len(loop_df) > 10:
        pnl_ch = loop_df['pnl'].diff().dropna()
        print(f"Avg PnL change/row: {pnl_ch.mean():.2f}")
        print(f"Worst single-row PnL drop: {pnl_ch.min():.2f}")
        print(f"Best single-row PnL gain: {pnl_ch.max():.2f}")
        print(f"PnL volatility (std): {pnl_ch.std():.2f}")
    print()

    if 'mode' in loop_df.columns:
        print('MODE DISTRIBUTION:')
        mode_counts = loop_df['mode'].value_counts()
        for m, c in mode_counts.items():
            print(f"  {m}: {c} ({c/len(loop_df)*100:.1f}%)")
    print()

    if 'is_aggressive' in loop_df.columns:
        agg = loop_df['is_aggressive'].sum()
        pas = len(loop_df) - agg
        print(f"Aggressive actions: {agg} ({agg/len(loop_df)*100:.1f}%)")
        print(f"Passive actions: {pas} ({pas/len(loop_df)*100:.1f}%)")
    print()

    # Order stats
    if not order_df.empty:
        submit_orders = order_df[order_df['event_type'].str.startswith('submit')]
        cancel_orders = order_df[order_df['event_type'] == 'cancel_all']
        print(f"Orders submitted: {len(submit_orders)}")
        print(f"Cancel-alls: {len(cancel_orders)}")
        if not submit_orders.empty:
            agg_sub = submit_orders[submit_orders['event_type'] == 'submit_agg']
            pas_sub = submit_orders[submit_orders['event_type'] == 'submit_pas']
            print(f"  Aggressive submits: {len(agg_sub)}")
            print(f"  Passive submits: {len(pas_sub)}")
    print()

    if not fill_df.empty:
        print('FILL ANALYSIS:')
        print(f"  Total fills: {len(fill_df)}")
        print(f"  Total shares traded: {fill_df['qty'].sum()}")
        print(f"  Avg edge per share: {fill_df['edge_per_share'].mean():.4f}")
        print(f"  Total estimated edge: {fill_df['est_edge_dollars'].sum():.2f}")
        print(f"  Pct fills with positive edge: {(fill_df['edge_per_share'] > 0).mean()*100:.1f}%")

        print('\n  EDGE BY MODE:')
        mode_edge = fill_df.groupby('mode').agg(
            fills=('qty', 'count'),
            shares=('qty', 'sum'),
            avg_edge=('edge_per_share', 'mean'),
            total_edge=('est_edge_dollars', 'sum')
        ).sort_values('shares', ascending=False)
        for idx, row in mode_edge.iterrows():
            print(f"    {idx}: {row['fills']} fills, {row['shares']:.0f} shares, "
                  f"avg_edge={row['avg_edge']:.4f}, total_edge={row['total_edge']:.2f}")
    print()

    if 'mid' in loop_df.columns and len(loop_df) > 20:
        high_inv = loop_df[loop_df['inventory'].abs() > 10000]
        if not high_inv.empty:
            mid_range = high_inv['mid'].max() - high_inv['mid'].min()
            print(f"Mid price range during high-inv periods: {mid_range:.4f}")
            print(f"  (Lower = less self-impact)")
    print()

    # Charts
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle('LT2 v6 Trade Analysis', fontsize=16, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(loop_df['tick'], loop_df['pnl'], linewidth=0.8)
    ax.axvline(x=second_block_time, color='gray', linestyle='--', alpha=0.5, label='Block 2')
    ax.set_title('PnL over Tick')
    ax.set_xlabel('Tick')
    ax.set_ylabel('PnL')
    ax.legend()

    ax = axes[0, 1]
    ax.plot(loop_df['tick'], loop_df['inventory'], linewidth=0.8)
    ax.axvline(x=second_block_time, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=0, color='red', linestyle='-', alpha=0.3)
    ax.set_title('Inventory over Tick')
    ax.set_xlabel('Tick')
    ax.set_ylabel('Inventory')

    ax = axes[1, 0]
    ax.plot(loop_df['tick'], loop_df['mid'], linewidth=0.8, label='Mid')
    ax.axvline(x=second_block_time, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('Mid Price over Tick')
    ax.set_xlabel('Tick')
    ax.set_ylabel('Price')
    ax.legend()

    ax = axes[1, 1]
    ax.plot(loop_df['tick'], loop_df['spread'], linewidth=0.6, alpha=0.7)
    ax.axvline(x=second_block_time, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('Spread over Tick')
    ax.set_xlabel('Tick')
    ax.set_ylabel('Spread')

    ax = axes[2, 0]
    if not fill_df.empty:
        edges = fill_df['edge_per_share'].dropna()
        if len(edges) > 0:
            ax.hist(edges, bins=30, edgecolor='black', alpha=0.7)
            ax.axvline(x=0, color='red', linestyle='--')
            ax.axvline(x=edges.mean(), color='blue', linestyle='--', label=f'Mean={edges.mean():.4f}')
            ax.legend()
    ax.set_title('Edge per Share Distribution')
    ax.set_xlabel('Edge')
    ax.set_ylabel('Count')

    ax = axes[2, 1]
    ax.plot(loop_df['tick'], loop_df['urgency'], linewidth=0.6, alpha=0.7)
    ax.axvline(x=second_block_time, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('Urgency over Tick')
    ax.set_xlabel('Tick')
    ax.set_ylabel('Urgency')

    plt.tight_layout()

    if save_prefix:
        fig.savefig(f'{save_prefix}_analysis.png', dpi=150, bbox_inches='tight')
        print(f"Chart saved: {save_prefix}_analysis.png")

    plt.close(fig)


def save_run_logs(loop_df, order_df, fill_df, prefix='lt2_run'):
    loop_df.to_csv(f'{prefix}_loop.csv', index=False)
    order_df.to_csv(f'{prefix}_orders.csv', index=False)
    fill_df.to_csv(f'{prefix}_fills.csv', index=False)
    print(f"Saved: {prefix}_loop.csv, {prefix}_orders.csv, {prefix}_fills.csv")

In [8]:
# =========================================================
# Helper functions
# =========================================================


def get_best_bid_ask(book):
    bids = book.get('bids', [])
    asks = book.get('asks', [])
    best_bid = float(bids[0]['price']) if bids else None
    best_ask = float(asks[0]['price']) if asks else None
    return best_bid, best_ask


def estimate_pnl(sec, mid_price):
    if sec is None:
        return 0.0
    if 'realized' in sec and 'unrealized' in sec:
        try:
            return float(sec['realized']) + float(sec['unrealized'])
        except Exception:
            pass
    return float(sec.get('position', 0)) * mid_price


def normalize_order(order):
    if order is None:
        return None
    qty = int(order['quantity'] / 100) * 100
    if qty < 100:
        return None
    return {
        'action': order['action'],
        'quantity': qty,
        'price': round(float(order['price']), 2)
    }


def cap_order(order, inventory, position_limit=50000, open_buy=0, open_sell=0):
    """
    Cap order to avoid gross limit violations.

    CRITICAL FIX (v6): Orders that REDUCE abs(inventory) are always safe.
    v5 bug: treated all orders as potentially increasing exposure,
    which blocked every unwind order when starting at the position limit.
    """
    if order is None:
        return None

    qty = order['quantity']

    # Determine if this order is unwinding (reducing abs inventory)
    is_unwinding = (
        (inventory > 0 and order['action'] == 'SELL') or
        (inventory < 0 and order['action'] == 'BUY')
    )

    if is_unwinding:
        # Unwind orders are always safe — just cap to actual inventory
        qty = min(qty, abs(inventory))
    else:
        # Adding to position: check gross limits strictly
        gross_used = abs(inventory) + open_buy + open_sell
        gross_remaining = max(0, position_limit - gross_used)
        qty = min(qty, gross_remaining)

    # Don't flip sides beyond a small buffer
    qty = min(qty, abs(inventory) + 500)
    qty = int(qty / 100) * 100

    if qty < 100:
        return None
    order['quantity'] = qty
    return order


def should_replace(current, desired, cur_mode, des_mode, tick, placed_tick):
    """Decide whether to cancel and replace a working passive order."""
    if desired is None or current is None:
        return True
    if cur_mode != des_mode:
        return True
    if current['action'] != desired['action']:
        return True
    if abs(current['price'] - desired['price']) >= 0.02:
        return True
    if abs(current['quantity'] - desired['quantity']) >= 300:
        return True
    # Refresh stale orders periodically
    if tick - placed_tick >= 3:
        return True
    return False

In [9]:
def run_live(gateway, trader, ui, ticker='GTA', poll_delay=0.25, debug_every=5):
    """
    Main trading loop.

    Each iteration:
    1. Read market state (book, position, PnL)
    2. Get trading decision from Trader
    3. Cap order for gross limits
    4. Execute: cancel stale orders, submit new ones
    5. Log everything for post-trade analysis
    """
    ui.set_gateway(gateway)

    loop_log, order_log, fill_log = [], [], []
    prev_inv, prev_mid, prev_mode = None, None, None
    working_order, working_mode, working_tick = None, None, None

    print(f'[START] LT2 v6 live trader for {ticker}')

    while True:
        try:
            if ui.stop_requested:
                print('[STOP] UI stop requested')
                try:
                    gateway.cancel_all_orders()
                except Exception:
                    pass
                break

            case = gateway.get_case()
            tick = case.get('tick')
            status = case.get('status')

            if status != 'ACTIVE':
                print(f'[END] status={status} tick={tick}')
                break

            book = gateway.get_book(ticker)
            best_bid, best_ask = get_best_bid_ask(book)

            if best_bid is None or best_ask is None:
                time.sleep(poll_delay)
                continue

            bid_sizes = [l['quantity'] - l.get('quantity_filled', 0) for l in book.get('bids', [])]
            ask_sizes = [l['quantity'] - l.get('quantity_filled', 0) for l in book.get('asks', [])]

            sec = gateway.get_security(ticker)
            inventory = int(float(sec.get('position', 0))) if sec else 0
            mid = (best_bid + best_ask) / 2
            pnl = estimate_pnl(sec, mid)

            # Get decision
            decision = trader.decide(
                best_bid=best_bid, best_ask=best_ask,
                inventory=inventory, elapsed=tick,
                bid_sizes=bid_sizes, ask_sizes=ask_sizes
            )

            mode = decision['mode']
            is_agg = decision.get('is_aggressive', False)
            desired = normalize_order(decision.get('order'))

            # Cap to gross limits (v6: unwind orders pass through safely)
            open_buy, open_sell = gateway.get_open_order_quantity(ticker)
            desired = cap_order(desired, inventory, 50000, open_buy, open_sell)

            # TWAP gap for display
            twap_gap = 0
            if trader.twap and tick is not None:
                twap_gap = trader.twap.schedule_gap(inventory, tick)

            # Debug print
            if tick and debug_every and tick % debug_every == 0:
                bid_3 = sum(bid_sizes[:3])
                ask_3 = sum(ask_sizes[:3])
                des_str = f"{desired['action']} {desired['quantity']}@{desired['price']:.2f}" if desired else 'None'
                print(
                    f'[T{tick}] bid={best_bid:.2f} ask={best_ask:.2f} '
                    f'inv={inventory} pnl={pnl:.2f} '
                    f'mode={mode} urg={decision.get("urgency",0):.2f} '
                    f'twap_gap={twap_gap} depth={bid_3}/{ask_3} '
                    f'order={des_str}'
                )

            # Infer fills from inventory changes
            if prev_inv is not None and prev_mid is not None and prev_mode is not None:
                infer_and_log_fills(fill_log, tick, prev_inv, inventory, best_bid, best_ask, prev_mid, prev_mode)

            if prev_inv is not None and inventory != prev_inv:
                print(f'[FILL] tick={tick} {prev_inv} -> {inventory} (delta={inventory - prev_inv})')
                working_order, working_mode, working_tick = None, None, None

            # === Order execution ===
            if inventory == 0:
                # Flat: cancel everything
                if working_order is not None:
                    try:
                        gateway.cancel_all_orders()
                    except Exception:
                        pass
                    working_order, working_mode, working_tick = None, None, None

            else:
                if is_agg:
                    # AGGRESSIVE: always cancel first, then submit fresh
                    try:
                        gateway.cancel_all_orders()
                        log_order(order_log, tick, ticker, 'cancel_all', mode, None, None, None, None, inventory)
                    except Exception:
                        pass
                    working_order, working_mode, working_tick = None, None, None

                    if desired and desired['quantity'] >= 100:
                        time.sleep(0.05)  # Brief pause to let cancel propagate
                        try:
                            gateway.submit_order(ticker, 'LIMIT', desired['quantity'], desired['action'], desired['price'])
                            log_order(order_log, tick, ticker, 'submit_agg', mode, 'LIMIT',
                                      desired['action'], desired['quantity'], desired['price'], inventory)
                            print(f"[AGG] tick={tick} {desired['action']} {desired['quantity']} @ {desired['price']:.2f} mode={mode}")
                        except Exception as e:
                            print(f'[WARN] tick={tick} agg submit failed: {e}')

                else:
                    # PASSIVE: cancel and replace only when needed
                    needs_replace = should_replace(working_order, desired, working_mode, mode, tick, working_tick or 0)

                    if needs_replace:
                        try:
                            gateway.cancel_all_orders()
                            log_order(order_log, tick, ticker, 'cancel_all', mode, None, None, None, None, inventory)
                        except Exception:
                            pass
                        working_order, working_mode, working_tick = None, None, None

                        if desired and desired['quantity'] >= 100:
                            time.sleep(0.05)
                            try:
                                gateway.submit_order(ticker, 'LIMIT', desired['quantity'], desired['action'], desired['price'])
                                log_order(order_log, tick, ticker, 'submit_pas', mode, 'LIMIT',
                                          desired['action'], desired['quantity'], desired['price'], inventory)
                                print(f"[PAS] tick={tick} {desired['action']} {desired['quantity']} @ {desired['price']:.2f} mode={mode}")
                                working_order = desired
                                working_mode = mode
                                working_tick = tick
                            except Exception as e:
                                print(f'[WARN] tick={tick} passive submit failed: {e}')

            # Log
            log_loop(loop_log, tick, best_bid, best_ask, inventory, pnl, decision)
            prev_inv, prev_mid, prev_mode = inventory, mid, mode

            # UI update
            bid_3 = sum(bid_sizes[:3])
            ask_3 = sum(ask_sizes[:3])
            ui.update(
                inventory=inventory, pnl=pnl, phase=decision.get('phase', '-'),
                mode=mode, best_bid=best_bid, best_ask=best_ask,
                bid_depth_3=bid_3, ask_depth_3=ask_3,
                urgency=decision.get('urgency', 0), imbalance=decision.get('imbalance', 0),
                twap_gap=twap_gap, working_order=working_order,
                alert_text=decision.get('alert_text', 'NORMAL'),
                alert_bg=decision.get('alert_bg', 'dark green')
            )

            time.sleep(poll_delay)

        except Exception as e:
            print(f'[ERROR] tick={tick}: {e}')
            time.sleep(poll_delay)

    loop_df = pd.DataFrame(loop_log)
    order_df = pd.DataFrame(order_log)
    fill_df = pd.DataFrame(fill_log)
    print(f'[DONE] loop={len(loop_df)}, orders={len(order_df)}, fills={len(fill_df)}')
    return loop_df, order_df, fill_df

In [12]:
# === CONFIGURE HERE ===
USERNAME = 'user4'
PASSWORD = 'user4'
HOST = 'http://141.211.79.101:10002'
TICKER = 'GTA'
POLL_DELAY = 0.25  # v6: slightly faster polling for better responsiveness

gateway = RITGateway(USERNAME, PASSWORD, host=HOST)

monitor = MarketMonitor(
    max_history=60,
    wide_spread_threshold=0.06,
    thin_depth_threshold=1500,
    imbalance_threshold=0.55,
    depth_surge_multiplier=2.0,
)

twap = TWAPScheduler(
    start_tick=0,
    end_tick=110,           # v6: finish 10 ticks before block 2 for safety margin
    catchup_sensitivity=1.2,
    min_progress_buffer=1500,
)

trader = Trader(
    monitor=monitor,
    twap=twap,
    position_limit=50000,
    second_block_time=120,
    hard_flatten_time=225,
    final_flatten_time=275,  # v6: start final flatten a bit earlier
    pre_second_safe_inventory=2000,
    pre_second_window_1=30,
    pre_second_window_2=15,
    pre_second_window_3=5,
    patience_ticks=3,       # v6: shorter patience — start trading sooner
    wide_spread_threshold=0.06,
)

In [15]:
ui = TraderUI()

loop_df, order_df, fill_df = run_live(
    gateway=gateway,
    trader=trader,
    ui=ui,
    ticker=TICKER,
    poll_delay=POLL_DELAY,
    debug_every=5
)

[START] LT2 v6 live trader for GTA
[PAS] tick=1 SELL 400 @ 10.08 mode=Patience
[PAS] tick=4 SELL 400 @ 10.08 mode=Patience
[T5] bid=10.06 ask=10.08 inv=50000 pnl=9500.00 mode=Patience urg=0.20 twap_gap=-100 depth=2680.0/2300.0 order=SELL 400@10.08
[T5] bid=10.07 ask=10.08 inv=49900 pnl=10500.00 mode=Patience urg=0.20 twap_gap=-200 depth=2315.0/2300.0 order=SELL 400@10.08
[FILL] tick=5 50000 -> 49900 (delta=-100)
[PAS] tick=5 SELL 400 @ 10.08 mode=Patience
[FILL] tick=6 49900 -> 49500 (delta=-400)
[PAS] tick=6 SELL 400 @ 10.11 mode=Patience
[FILL] tick=7 49500 -> 49100 (delta=-400)
[PAS] tick=7 SELL 400 @ 10.19 mode=Patience
[T10] bid=10.18 ask=10.19 inv=49100 pnl=16404.00 mode=Patience urg=0.20 twap_gap=-1000 depth=92950.0/1800.0 order=SELL 400@10.19
[PAS] tick=10 SELL 400 @ 10.19 mode=Patience
[T10] bid=10.18 ask=10.19 inv=49100 pnl=16404.00 mode=Patience urg=0.20 twap_gap=-1000 depth=92800.0/1800.0 order=SELL 400@10.19
[T10] bid=10.20 ask=10.21 inv=48700 pnl=17374.00 mode=Patience ur

In [16]:
analyze_run(loop_df, order_df, fill_df, second_block_time=120, save_prefix='lt2_v6')
save_run_logs(loop_df, order_df, fill_df, prefix='lt2_v6')

                    RUN ANALYSIS
Final tick: 239
Final PnL: 18956.28
Final inventory: 38
Max abs inventory: 50070
Avg abs inventory: 30049
Time with |inv| > 3000: 505 rows

Pre-block final PnL: 14376.44
Pre-block max abs inv: 50000
Post-block final PnL: 18956.28
Post-block max abs inv: 50070
Inv at second block: 70
PnL at second block: 14374.34

Avg spread: 0.0534
Max spread: 0.8800
Pct time spread > 0.10: 11.2%

Avg PnL change/row: 20.94
Worst single-row PnL drop: -12482.40
Best single-row PnL gain: 15014.00
PnL volatility (std): 1389.00

MODE DISTRIBUTION:
  Patience: 279 (48.8%)
  Passive Unwind: 201 (35.1%)
  Wide Spread Passive: 46 (8.0%)
  Liquidity Surge: 17 (3.0%)
  Favorable Patient: 15 (2.6%)
  Pre-Block Forced: 12 (2.1%)
  Emergency Sell: 1 (0.2%)
  Opportunistic Sell: 1 (0.2%)

Aggressive actions: 31 (5.4%)
Passive actions: 541 (94.6%)

Orders submitted: 210
Cancel-alls: 268
  Aggressive submits: 31
  Passive submits: 179

FILL ANALYSIS:
  Total fills: 126
  Total shares tr